# Employee Attrition Prediction with XGBoost (Accuracy-First)

This notebook is optimized to maximize **test accuracy** while preserving transparent generalization and minority-class diagnostics.

## Objectives
- Primary objective: maximize classification **accuracy**.
- Guardrail metrics: F1, recall, precision, ROC-AUC, PR-AUC.
- Track **runtime vs performance** for each hyperparameter trial.
- Add full chart pack for distribution, variance, parameters, thresholds, and errors.


## 0) Setup and Reproducibility


In [ ]:
%pip install pandas numpy scikit-learn xgboost matplotlib seaborn joblib


In [ ]:
from pathlib import Path
import json
import time
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from IPython.display import display
from sklearn.model_selection import train_test_split, RandomizedSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    average_precision_score,
    classification_report,
    ConfusionMatrixDisplay,
)
from xgboost import XGBClassifier

RANDOM_STATE = 42
SELECTION_OBJECTIVE = 'accuracy'
np.random.seed(RANDOM_STATE)
warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')

print('RANDOM_STATE =', RANDOM_STATE)
print('SELECTION_OBJECTIVE =', SELECTION_OBJECTIVE)


## 1) Load Data and Validate Contracts


In [ ]:
def find_project_root(start: Path) -> Path:
    for candidate in (start, *start.parents):
        if (candidate / 'outputs').exists() and (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError("Could not find project root with 'data/' and 'outputs/'.")


def ensure_required_files(root: Path, rel_paths: list[str]) -> dict[str, Path]:
    resolved = {}
    missing = []
    for rel in rel_paths:
        p = root / rel
        resolved[rel] = p
        if not p.exists():
            missing.append(str(p))
    if missing:
        raise FileNotFoundError('Missing required files:\n' + '\n'.join(missing))
    return resolved


def convert_boolean_like_strings(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    out = df.copy()
    converted = []
    for col in out.columns:
        if out[col].dtype == 'object':
            values = set(out[col].dropna().astype(str).str.strip().str.lower().unique())
            if values and values.issubset({'true', 'false'}):
                out[col] = out[col].astype(str).str.strip().str.lower().map({'true': 1, 'false': 0}).astype('int8')
                converted.append(col)
    return out, converted


PROJECT_ROOT = find_project_root(Path.cwd())
print('PROJECT_ROOT:', PROJECT_ROOT)

required_files = [
    'outputs/preprocessed_data.csv',
    'outputs/X_train_smote.csv',
    'outputs/X_test.csv',
    'outputs/y_train_smote.csv',
    'outputs/y_test.csv',
]
paths = ensure_required_files(PROJECT_ROOT, required_files)

preprocessed_df = pd.read_csv(paths['outputs/preprocessed_data.csv'])
X_train_smote = pd.read_csv(paths['outputs/X_train_smote.csv'])
X_test = pd.read_csv(paths['outputs/X_test.csv'])
y_train_smote_df = pd.read_csv(paths['outputs/y_train_smote.csv'])
y_test_df = pd.read_csv(paths['outputs/y_test.csv'])

if list(y_train_smote_df.columns) != ['Attrition'] or list(y_test_df.columns) != ['Attrition']:
    raise ValueError("Target columns in y files must be exactly ['Attrition'].")

if list(X_train_smote.columns) != list(X_test.columns):
    raise ValueError('X_train_smote and X_test feature columns must match exactly.')

if X_train_smote.shape[1] != 44:
    raise ValueError(f'Expected 44 feature columns, found {X_train_smote.shape[1]}.')

leak_cols = [c for c in X_train_smote.columns if 'attrition' in c.lower()]
if leak_cols:
    raise ValueError(f'Potential leakage columns in X: {leak_cols}')

X_train_smote, converted_train_cols = convert_boolean_like_strings(X_train_smote)
X_test, converted_test_cols = convert_boolean_like_strings(X_test)

for c in X_test.columns:
    if X_train_smote[c].dtype == 'object':
        X_train_smote[c] = pd.to_numeric(X_train_smote[c], errors='raise')
    if X_test[c].dtype == 'object':
        X_test[c] = pd.to_numeric(X_test[c], errors='raise')

y_train_smote = y_train_smote_df['Attrition'].astype(int)
y_test = y_test_df['Attrition'].astype(int)

# Reconstruct original imbalanced split to improve real-world generalization
full_df = preprocessed_df.copy()
if full_df['Attrition'].dtype == 'object':
    full_df['Attrition'] = full_df['Attrition'].map({'Yes': 1, 'No': 0})
full_df['Attrition'] = full_df['Attrition'].astype(int)

X_full = pd.get_dummies(full_df.drop(columns='Attrition'), drop_first=True)
y_full = full_df['Attrition']

X_train_raw, X_test_raw, y_train_raw, y_test_raw = train_test_split(
    X_full,
    y_full,
    test_size=0.20,
    stratify=y_full,
    random_state=RANDOM_STATE,
)

X_train_raw = X_train_raw.reindex(columns=X_test.columns, fill_value=0)
X_test_raw = X_test_raw.reindex(columns=X_test.columns, fill_value=0)

print('Shapes:')
print('- preprocessed_df:', preprocessed_df.shape)
print('- X_train_smote:', X_train_smote.shape, '| y_train_smote:', y_train_smote.shape)
print('- X_train_raw:', X_train_raw.shape, '| y_train_raw:', y_train_raw.shape)
print('- X_test official:', X_test.shape, '| y_test official:', y_test.shape)
print('Converted bool-like columns:', sorted(set(converted_train_cols + converted_test_cols)))


## 2) Dataset Distribution, Variance, and Relationship Charts


In [ ]:
# Target distribution chart for full/train/test
class_df = pd.DataFrame({
    'split': ['full', 'full', 'train_raw', 'train_raw', 'train_smote', 'train_smote', 'test', 'test'],
    'class': [0, 1, 0, 1, 0, 1, 0, 1],
    'count': [
        int((full_df['Attrition'] == 0).sum()), int((full_df['Attrition'] == 1).sum()),
        int((y_train_raw == 0).sum()), int((y_train_raw == 1).sum()),
        int((y_train_smote == 0).sum()), int((y_train_smote == 1).sum()),
        int((y_test == 0).sum()), int((y_test == 1).sum()),
    ]
})
class_df['pct'] = class_df.groupby('split')['count'].transform(lambda x: (x / x.sum()) * 100)

display(class_df)

plt.figure(figsize=(10, 4))
sns.barplot(data=class_df, x='split', y='pct', hue='class')
plt.title('Target Distribution by Split')
plt.ylabel('Percentage')
plt.ylim(0, 100)
plt.show()


In [ ]:
# Feature variance ranking and correlation heatmap
numeric_cols = [c for c in X_train_raw.columns if pd.api.types.is_numeric_dtype(X_train_raw[c])]
variance_series = X_train_raw[numeric_cols].var().sort_values(ascending=False)

variance_df = variance_series.head(20).reset_index()
variance_df.columns = ['feature', 'variance']
display(variance_df)

plt.figure(figsize=(10, 6))
sns.barplot(data=variance_df, y='feature', x='variance', color='steelblue')
plt.title('Top 20 Feature Variances (Training Raw)')
plt.xlabel('Variance')
plt.ylabel('Feature')
plt.show()

corr_features = variance_series.head(15).index.tolist()
corr_matrix = X_train_raw[corr_features].corr()

plt.figure(figsize=(12, 8))
sns.heatmap(corr_matrix, cmap='coolwarm', center=0, square=False)
plt.title('Correlation Heatmap (Top Variance Features)')
plt.show()


In [ ]:
# Train vs test distribution overlays for top numeric features
overlay_features = variance_series.head(6).index.tolist()
fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, feature in zip(axes.ravel(), overlay_features):
    sns.kdeplot(X_train_raw[feature], ax=ax, label='train_raw', fill=True, alpha=0.35)
    sns.kdeplot(X_test[feature], ax=ax, label='test', fill=True, alpha=0.25)
    ax.set_title(feature)
    ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
# Box and violin plots for key features by attrition class
full_numeric = full_df.select_dtypes(include=[np.number]).drop(columns=['Attrition'])
key_features = full_numeric.var().sort_values(ascending=False).head(4).index.tolist()

plot_df = full_df[['Attrition'] + key_features].copy()
long_df = plot_df.melt(id_vars='Attrition', var_name='feature', value_name='value')

plt.figure(figsize=(14, 5))
sns.boxplot(data=long_df, x='feature', y='value', hue='Attrition')
plt.title('Boxplot: Key Features by Attrition Class')
plt.xticks(rotation=20)
plt.show()

plt.figure(figsize=(14, 5))
sns.violinplot(data=long_df, x='feature', y='value', hue='Attrition', split=False)
plt.title('Violin Plot: Key Features by Attrition Class')
plt.xticks(rotation=20)
plt.show()


## 3) Baseline Models


In [ ]:
def evaluate_from_probabilities(y_true: pd.Series, y_prob: np.ndarray, threshold: float) -> tuple[dict, np.ndarray]:
    y_pred = (y_prob >= threshold).astype(int)
    metrics = {
        'threshold': float(threshold),
        'f1': float(f1_score(y_true, y_pred)),
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'precision': float(precision_score(y_true, y_pred, zero_division=0)),
        'recall': float(recall_score(y_true, y_pred, zero_division=0)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
        'pr_auc': float(average_precision_score(y_true, y_prob)),
    }
    return metrics, y_pred


# Baseline A: SMOTE training split
baseline_smote = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
)
baseline_smote.fit(X_train_smote, y_train_smote)
baseline_smote_prob = baseline_smote.predict_proba(X_test)[:, 1]
baseline_smote_metrics, baseline_smote_pred = evaluate_from_probabilities(y_test, baseline_smote_prob, 0.50)

# Baseline B: Raw training split + class weight
neg_raw, pos_raw = np.bincount(y_train_raw)
base_scale_pos_weight = float(neg_raw / pos_raw)

baseline_raw_weighted = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
    scale_pos_weight=base_scale_pos_weight,
)
baseline_raw_weighted.fit(X_train_raw, y_train_raw)
baseline_raw_prob = baseline_raw_weighted.predict_proba(X_test)[:, 1]
baseline_raw_metrics, baseline_raw_pred = evaluate_from_probabilities(y_test, baseline_raw_prob, 0.50)

baseline_table = pd.DataFrame([
    {'model': 'Baseline_SMOTE_train', 'evaluation': 'threshold=0.50', **baseline_smote_metrics},
    {'model': 'Baseline_RawWeighted_train', 'evaluation': 'threshold=0.50', **baseline_raw_metrics},
])

display(baseline_table.round(4))
print('base_scale_pos_weight =', round(base_scale_pos_weight, 4))


## 4) Accuracy-First Two-Stage Hyperparameter Search


In [ ]:
# Split raw train into tuning-train and validation for threshold selection
X_tune, X_val, y_tune, y_val = train_test_split(
    X_train_raw,
    y_train_raw,
    test_size=0.20,
    stratify=y_train_raw,
    random_state=RANDOM_STATE,
)

neg_tune, pos_tune = np.bincount(y_tune)
scale_pos_weight_tune = float(neg_tune / pos_tune)

scoring_dict = {
    'accuracy': 'accuracy',
    'f1': 'f1',
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

xgb_base = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
)

param_dist_stage1 = {
    'n_estimators': [250, 350, 450, 600, 800, 1000],
    'max_depth': [2, 3, 4, 5, 6],
    'learning_rate': [0.01, 0.02, 0.03, 0.05, 0.08, 0.10],
    'min_child_weight': [1, 2, 3, 5, 7, 10],
    'subsample': [0.60, 0.70, 0.80, 0.90, 1.00],
    'colsample_bytree': [0.50, 0.60, 0.70, 0.80, 0.90, 1.00],
    'gamma': [0.0, 0.1, 0.2, 0.4, 0.8],
    'reg_alpha': [0.0, 0.01, 0.05, 0.1, 0.5, 1.0, 2.0],
    'reg_lambda': [1.0, 2.0, 3.0, 5.0, 8.0, 12.0],
    'scale_pos_weight': [
        round(scale_pos_weight_tune * 0.75, 4),
        round(scale_pos_weight_tune * 0.90, 4),
        round(scale_pos_weight_tune, 4),
        round(scale_pos_weight_tune * 1.10, 4),
        round(scale_pos_weight_tune * 1.25, 4),
    ],
}

stage1_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_stage1,
    n_iter=28,
    scoring=scoring_dict,
    refit='accuracy',
    n_jobs=-1,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    return_train_score=True,
)

stage1_t0 = time.perf_counter()
stage1_search.fit(X_tune, y_tune)
stage1_t1 = time.perf_counter()
stage1_wall_time_seconds = stage1_t1 - stage1_t0

print('Stage 1 best CV accuracy:', round(stage1_search.best_score_, 5))
print('Stage 1 best params:', stage1_search.best_params_)
print('Stage 1 wall time (s):', round(stage1_wall_time_seconds, 2))


In [ ]:
def refine_int(value: int, min_value: int, max_value: int, step: int = 1) -> list[int]:
    values = [value - 2 * step, value - step, value, value + step, value + 2 * step]
    values = [min(max(v, min_value), max_value) for v in values]
    return sorted(set(int(v) for v in values))


def refine_float(value: float, min_value: float, max_value: float, mult=(0.70, 0.85, 1.00, 1.15, 1.30), ndigits: int = 4) -> list[float]:
    values = [value * m for m in mult]
    values = [min(max(v, min_value), max_value) for v in values]
    return sorted(set(round(float(v), ndigits) for v in values))


best1 = stage1_search.best_params_

param_dist_stage2 = {
    'n_estimators': refine_int(best1['n_estimators'], 200, 1500, step=100),
    'max_depth': refine_int(best1['max_depth'], 2, 10, step=1),
    'learning_rate': refine_float(best1['learning_rate'], 0.005, 0.20),
    'min_child_weight': refine_int(best1['min_child_weight'], 1, 20, step=1),
    'subsample': refine_float(best1['subsample'], 0.50, 1.00),
    'colsample_bytree': refine_float(best1['colsample_bytree'], 0.40, 1.00),
    'gamma': sorted(set([0.0] + refine_float(best1['gamma'], 0.0, 3.0))),
    'reg_alpha': sorted(set([0.0] + refine_float(best1['reg_alpha'], 0.0, 5.0))),
    'reg_lambda': refine_float(best1['reg_lambda'], 0.5, 20.0),
    'scale_pos_weight': refine_float(best1['scale_pos_weight'], 1.0, 20.0),
}

stage2_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_dist_stage2,
    n_iter=16,
    scoring=scoring_dict,
    refit='accuracy',
    n_jobs=-1,
    cv=cv,
    verbose=1,
    random_state=RANDOM_STATE,
    return_train_score=True,
)

stage2_t0 = time.perf_counter()
stage2_search.fit(X_tune, y_tune)
stage2_t1 = time.perf_counter()
stage2_wall_time_seconds = stage2_t1 - stage2_t0

print('Stage 2 best CV accuracy:', round(stage2_search.best_score_, 5))
print('Stage 2 best params:', stage2_search.best_params_)
print('Stage 2 wall time (s):', round(stage2_wall_time_seconds, 2))

cv_summary = pd.DataFrame([
    {
        'stage': 'stage1',
        'best_cv_accuracy': stage1_search.best_score_,
        'best_cv_f1': stage1_search.cv_results_['mean_test_f1'][stage1_search.best_index_],
        'wall_time_seconds': stage1_wall_time_seconds,
    },
    {
        'stage': 'stage2',
        'best_cv_accuracy': stage2_search.best_score_,
        'best_cv_f1': stage2_search.cv_results_['mean_test_f1'][stage2_search.best_index_],
        'wall_time_seconds': stage2_wall_time_seconds,
    },
])
display(cv_summary.round(4))


## 5) Runtime Analytics and Hyperparameter Impact Charts


In [ ]:
def search_results_to_trial_df(search_obj, stage_name: str) -> pd.DataFrame:
    cvres = search_obj.cv_results_
    n = len(cvres['params'])
    rows = []
    for i in range(n):
        row = {
            'stage': stage_name,
            'trial_index': i,
            'trial_id': f'{stage_name}_{i+1:03d}',
            'rank_accuracy': int(cvres['rank_test_accuracy'][i]),
            'mean_fit_time': float(cvres['mean_fit_time'][i]),
            'std_fit_time': float(cvres['std_fit_time'][i]),
            'mean_cv_accuracy': float(cvres['mean_test_accuracy'][i]),
            'std_cv_accuracy': float(cvres['std_test_accuracy'][i]),
            'mean_cv_f1': float(cvres['mean_test_f1'][i]),
            'std_cv_f1': float(cvres['std_test_f1'][i]),
        }
        params = cvres['params'][i]
        for k, v in params.items():
            row[k] = v
        rows.append(row)
    df = pd.DataFrame(rows)
    df['is_selected_model'] = False
    df['selection_reason'] = ''
    return df


stage1_trials = search_results_to_trial_df(stage1_search, 'stage1')
stage2_trials = search_results_to_trial_df(stage2_search, 'stage2')
stage2_trials.loc[stage2_trials['rank_accuracy'] == 1, 'is_selected_model'] = True
stage2_trials.loc[stage2_trials['rank_accuracy'] == 1, 'selection_reason'] = 'best_refit_accuracy'

runtime_trials_df = pd.concat([stage1_trials, stage2_trials], ignore_index=True)
runtime_trials_df['cumulative_mean_fit_time'] = runtime_trials_df['mean_fit_time'].cumsum()
runtime_trials_df['cumulative_best_cv_accuracy'] = runtime_trials_df['mean_cv_accuracy'].cummax()

display(runtime_trials_df.head(10))

# Runtime per trial vs CV accuracy
plt.figure(figsize=(9, 5))
sns.scatterplot(data=runtime_trials_df, x='mean_fit_time', y='mean_cv_accuracy', hue='stage', style='is_selected_model', s=80)
plt.title('Runtime per Trial vs CV Accuracy')
plt.xlabel('Mean Fit Time (seconds)')
plt.ylabel('Mean CV Accuracy')
plt.show()

# Cumulative best accuracy vs cumulative runtime
plt.figure(figsize=(9, 5))
plt.plot(runtime_trials_df['cumulative_mean_fit_time'], runtime_trials_df['cumulative_best_cv_accuracy'], color='tab:green')
plt.title('Cumulative Best CV Accuracy vs Cumulative Runtime')
plt.xlabel('Cumulative Mean Fit Time (seconds)')
plt.ylabel('Best CV Accuracy So Far')
plt.show()

# Hyperparameter impact charts
for param in ['max_depth', 'learning_rate', 'n_estimators']:
    plt.figure(figsize=(8, 4))
    sns.scatterplot(data=runtime_trials_df, x=param, y='mean_cv_accuracy', hue='stage', alpha=0.8)
    plt.title(f'Hyperparameter Impact: {param} vs CV Accuracy')
    plt.ylabel('Mean CV Accuracy')
    plt.show()

# 2D heatmap for strongest pair
pivot_df = runtime_trials_df.pivot_table(
    index='max_depth',
    columns='learning_rate',
    values='mean_cv_accuracy',
    aggfunc='mean',
)
plt.figure(figsize=(10, 6))
sns.heatmap(pivot_df, cmap='viridis')
plt.title('CV Accuracy Heatmap: max_depth vs learning_rate')
plt.show()

# Save trial log artifact
metrics_dir = PROJECT_ROOT / 'outputs' / 'metrics'
metrics_dir.mkdir(parents=True, exist_ok=True)
trial_log_path = metrics_dir / 'xgboost_runtime_trials.csv'
runtime_trials_df.to_csv(trial_log_path, index=False)
print('Saved trial log:', trial_log_path)


## 6) Threshold Selection (Validation Only)


In [ ]:
best_params = stage2_search.best_params_

threshold_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
    **best_params,
)
threshold_model.fit(X_tune, y_tune)

val_prob = threshold_model.predict_proba(X_val)[:, 1]
thresholds = np.round(np.arange(0.05, 0.96, 0.01), 2)

threshold_rows = []
for t in thresholds:
    pred = (val_prob >= t).astype(int)
    threshold_rows.append({
        'threshold': float(t),
        'accuracy': accuracy_score(y_val, pred),
        'f1': f1_score(y_val, pred),
        'precision': precision_score(y_val, pred, zero_division=0),
        'recall': recall_score(y_val, pred, zero_division=0),
    })

threshold_df = pd.DataFrame(threshold_rows)
best_threshold_accuracy = float(threshold_df.loc[threshold_df['accuracy'].idxmax(), 'threshold'])
best_threshold_f1 = float(threshold_df.loc[threshold_df['f1'].idxmax(), 'threshold'])

print('Selected threshold (accuracy):', best_threshold_accuracy)
print('Selected threshold (f1):', best_threshold_f1)

display(threshold_df.sort_values('accuracy', ascending=False).head(12))

plt.figure(figsize=(10, 5))
plt.plot(threshold_df['threshold'], threshold_df['accuracy'], label='Validation Accuracy', color='tab:green')
plt.plot(threshold_df['threshold'], threshold_df['f1'], label='Validation F1', color='tab:blue')
plt.plot(threshold_df['threshold'], threshold_df['recall'], label='Validation Recall', color='tab:orange')
plt.axvline(best_threshold_accuracy, color='tab:green', linestyle='--', label=f'Best Accuracy t={best_threshold_accuracy:.2f}')
plt.axvline(best_threshold_f1, color='tab:blue', linestyle='--', label=f'Best F1 t={best_threshold_f1:.2f}')
plt.title('Threshold vs Accuracy/F1/Recall (Validation)')
plt.xlabel('Threshold')
plt.ylabel('Score')
plt.legend()
plt.show()


## 7) Final Test Evaluation


In [ ]:
# Train final model once on full raw training split (leak-free sequence)
final_model = XGBClassifier(
    objective='binary:logistic',
    eval_metric='logloss',
    random_state=RANDOM_STATE,
    n_jobs=-1,
    tree_method='hist',
    **best_params,
)
final_model.fit(X_train_raw, y_train_raw)

test_prob = final_model.predict_proba(X_test)[:, 1]

final_metrics_050, pred_050 = evaluate_from_probabilities(y_test, test_prob, 0.50)
final_metrics_acc_sel, pred_acc_sel = evaluate_from_probabilities(y_test, test_prob, best_threshold_accuracy)
final_metrics_f1_sel, pred_f1_sel = evaluate_from_probabilities(y_test, test_prob, best_threshold_f1)

results_table = pd.DataFrame([
    {'model': 'Baseline_SMOTE_train', 'evaluation': 'threshold=0.50', **baseline_smote_metrics},
    {'model': 'Baseline_RawWeighted_train', 'evaluation': 'threshold=0.50', **baseline_raw_metrics},
    {'model': 'Tuned_RawWeighted', 'evaluation': 'threshold=0.50', **final_metrics_050},
    {'model': 'Tuned_RawWeighted', 'evaluation': f'selected_accuracy_threshold={best_threshold_accuracy:.2f}', **final_metrics_acc_sel},
    {'model': 'Tuned_RawWeighted', 'evaluation': f'selected_f1_threshold={best_threshold_f1:.2f}', **final_metrics_f1_sel},
])

display(results_table.round(4))

print('Classification report (selected accuracy threshold):')
print(classification_report(y_test, pred_acc_sel, digits=4))

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
ConfusionMatrixDisplay.from_predictions(y_test, pred_050, ax=axes[0], cmap='Blues')
axes[0].set_title('Threshold 0.50')
ConfusionMatrixDisplay.from_predictions(y_test, pred_acc_sel, ax=axes[1], cmap='Greens')
axes[1].set_title(f'Selected Accuracy ({best_threshold_accuracy:.2f})')
ConfusionMatrixDisplay.from_predictions(y_test, pred_f1_sel, ax=axes[2], cmap='Oranges')
axes[2].set_title(f'Selected F1 ({best_threshold_f1:.2f})')
plt.tight_layout()
plt.show()


## 8) Generalization and Quality Gates


In [ ]:
cv_best_accuracy = float(stage2_search.best_score_)
cv_best_accuracy_std = float(stage2_search.cv_results_['std_test_accuracy'][stage2_search.best_index_])
cv_best_f1 = float(stage2_search.cv_results_['mean_test_f1'][stage2_search.best_index_])

quality_gate_primary_pass = final_metrics_acc_sel['accuracy'] >= baseline_raw_metrics['accuracy']
recall_drop = baseline_raw_metrics['recall'] - final_metrics_acc_sel['recall']

quality_gate_secondary_note = (
    'OK' if recall_drop <= 0.10 else 'WARNING: recall drop > 0.10 vs baseline_raw_weighted'
)

gap_table = pd.DataFrame([
    {'metric': 'CV best accuracy (stage2)', 'value': cv_best_accuracy},
    {'metric': 'CV best accuracy std (stage2)', 'value': cv_best_accuracy_std},
    {'metric': 'CV best f1 (stage2 best-accuracy model)', 'value': cv_best_f1},
    {'metric': 'Test accuracy @ selected accuracy threshold', 'value': final_metrics_acc_sel['accuracy']},
    {'metric': 'Test f1 @ selected accuracy threshold', 'value': final_metrics_acc_sel['f1']},
])

display(gap_table.round(4))
print('Primary gate (tuned selected accuracy >= baseline raw accuracy):', quality_gate_primary_pass)
print('Secondary gate (recall delta check):', quality_gate_secondary_note)
print('Recall delta (baseline_raw - tuned_selected_accuracy):', round(recall_drop, 4))


## 9) Explainability and Error Analysis


In [ ]:
booster = final_model.get_booster()
importance_gain = pd.Series(booster.get_score(importance_type='gain'), name='gain')
importance_weight = pd.Series(booster.get_score(importance_type='weight'), name='weight')


def remap_findex(series: pd.Series, columns: pd.Index) -> pd.Series:
    if series.empty:
        return series
    if series.index.to_series().str.match(r'^f\d+$').all():
        mapping = {f'f{i}': col for i, col in enumerate(columns)}
        series.index = series.index.map(lambda key: mapping.get(key, key))
    return series

importance_gain = remap_findex(importance_gain, X_train_raw.columns).sort_values(ascending=False)
importance_weight = remap_findex(importance_weight, X_train_raw.columns).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 7))
importance_gain.head(20).sort_values().plot(kind='barh', ax=axes[0], color='tab:blue', title='Top Features by Gain')
importance_weight.head(20).sort_values().plot(kind='barh', ax=axes[1], color='tab:orange', title='Top Features by Weight')
plt.tight_layout()
plt.show()

# Error analysis at selected accuracy threshold
analysis_df = pd.DataFrame({
    'y_true': y_test.values,
    'y_prob': test_prob,
    'y_pred_accuracy_selected': pred_acc_sel,
    'y_pred_f1_selected': pred_f1_sel,
}, index=X_test.index)

conditions = [
    (analysis_df['y_true'] == 1) & (analysis_df['y_pred_accuracy_selected'] == 0),
    (analysis_df['y_true'] == 0) & (analysis_df['y_pred_accuracy_selected'] == 1),
]
labels = ['False Negative', 'False Positive']
analysis_df['error_type_accuracy_selected'] = np.select(conditions, labels, default='Correct')

print('Error distribution (selected accuracy threshold):')
print(analysis_df['error_type_accuracy_selected'].value_counts())

display(analysis_df[analysis_df['error_type_accuracy_selected'] == 'False Negative'].sort_values('y_prob', ascending=False).head(10))
display(analysis_df[analysis_df['error_type_accuracy_selected'] == 'False Positive'].sort_values('y_prob', ascending=False).head(10))


## 10) Export Artifacts


In [ ]:
metrics_dir = PROJECT_ROOT / 'outputs' / 'metrics'
preds_dir = PROJECT_ROOT / 'outputs' / 'predictions'
models_dir = PROJECT_ROOT / 'outputs' / 'models'
for d in [metrics_dir, preds_dir, models_dir]:
    d.mkdir(parents=True, exist_ok=True)

runtime_summary = {
    'stage1_wall_time_seconds': float(stage1_wall_time_seconds),
    'stage2_wall_time_seconds': float(stage2_wall_time_seconds),
    'total_wall_time_seconds': float(stage1_wall_time_seconds + stage2_wall_time_seconds),
    'stage1_trials': int(len(stage1_trials)),
    'stage2_trials': int(len(stage2_trials)),
    'total_trials': int(len(runtime_trials_df)),
    'best_trial_mean_fit_time_seconds': float(runtime_trials_df.loc[runtime_trials_df['is_selected_model'], 'mean_fit_time'].iloc[0]),
}

metrics_payload = {
    'selection_objective': SELECTION_OBJECTIVE,
    'runtime_summary': runtime_summary,
    'cv_accuracy_best': cv_best_accuracy,
    'cv_accuracy_std_best': cv_best_accuracy_std,
    'selected_threshold_accuracy': best_threshold_accuracy,
    'selected_threshold_f1': best_threshold_f1,
    'best_params_stage2': best_params,
    'baseline_smote_threshold_0_50': baseline_smote_metrics,
    'baseline_raw_weighted_threshold_0_50': baseline_raw_metrics,
    'tuned_threshold_0_50': final_metrics_050,
    'tuned_threshold_accuracy_selected': final_metrics_acc_sel,
    'tuned_threshold_f1_selected': final_metrics_f1_sel,
    'quality_gate_primary_pass': bool(quality_gate_primary_pass),
    'quality_gate_secondary_note': quality_gate_secondary_note,
    'created_utc': pd.Timestamp.utcnow().isoformat(),
}

metrics_path = metrics_dir / 'xgboost_metrics.json'
with metrics_path.open('w', encoding='utf-8') as f:
    json.dump(metrics_payload, f, indent=2)

pred_export = pd.DataFrame({
    'index': X_test.index,
    'y_true': y_test.values,
    'y_prob': test_prob,
    'y_pred_0_50': pred_050,
    'y_pred_accuracy_selected': pred_acc_sel,
    'y_pred_f1_selected': pred_f1_sel,
    'error_type_accuracy_selected': analysis_df['error_type_accuracy_selected'].values,
})
preds_path = preds_dir / 'xgboost_test_predictions.csv'
pred_export.to_csv(preds_path, index=False)

model_path = models_dir / 'xgboost_attrition_model.joblib'
joblib.dump(final_model, model_path)

print('Saved metrics:', metrics_path)
print('Saved predictions:', preds_path)
print('Saved model:', model_path)
print('Saved trials:', metrics_dir / 'xgboost_runtime_trials.csv')

display(results_table.round(4))


## 11) Final Summary


In [ ]:
tuned_rows = results_table[results_table['model'] == 'Tuned_RawWeighted'].copy()
best_acc_row = tuned_rows.loc[tuned_rows['accuracy'].idxmax()]
best_f1_row = tuned_rows.loc[tuned_rows['f1'].idxmax()]

print('Best tuned accuracy row:', best_acc_row['evaluation'], '| accuracy =', round(best_acc_row['accuracy'], 4))
print('Best tuned f1 row:', best_f1_row['evaluation'], '| f1 =', round(best_f1_row['f1'], 4))
print('Best tuned roc_auc =', round(tuned_rows['roc_auc'].max(), 4))
print('Primary quality gate pass:', quality_gate_primary_pass)
print('Secondary quality note:', quality_gate_secondary_note)

print('\nReport guidance:')
print('- Use selected accuracy threshold row for headline accuracy.')
print('- Also report selected f1 threshold row for minority-class sensitivity discussion.')
print('- Include runtime-vs-accuracy charts to justify parameter/runtime tradeoffs.')
